In [30]:
from supabase import create_client
from dotenv import load_dotenv
import os
from faker import Faker
import random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

supabase = create_client(url, key)

In [17]:
auth = []
for i in range(100):
    email = f"user{i}@uniandes.edu.co"
    password = f"Password{i}!"

    response = supabase.auth.admin.create_user({
        "email": email,
        "password": password,
        "email_confirm": True
    })
    
    auth.append(response.user.id)
   

In [18]:
fake = Faker()

users = []

for i in range(100):
    users.append({
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "zone_id": random.randint(1, 3),
        "auth_id": auth[i]
    })

df = pd.DataFrame(users)

supabase.table("users").insert(users).execute()

print(df)

    first_name last_name  zone_id                               auth_id
0       Amanda      Wong        1  8000c285-39a5-4d6d-8261-3f8450fc05f0
1        David   Perkins        3  76e5f133-ed2a-489c-af6c-67c49c233c2c
2        Julie     Scott        3  91dd48ec-9868-415b-ad0b-9adaf22ef666
3        Tonya     Jones        2  51dc2159-ba4e-46c3-b0f6-6244a0441bf7
4        Susan    Gordon        2  b07d63ee-3225-4ffe-9990-9fc1d902be87
..         ...       ...      ...                                   ...
95      Robert  Bradford        3  d3f47e23-66da-434e-876b-39aa3ad03a93
96     Annette    Murray        3  75a65f7f-a35d-4914-97d7-353587bbd701
97      Debbie     Stein        2  39454c0b-42e3-482c-bfc0-83bd69fb36a1
98  Jacqueline  Cisneros        2  1e4c0d2b-4e5a-4602-a9df-641a6e01d7a9
99    Patricia  Phillips        2  59fa20eb-d103-4bb5-b772-9881606fbce3

[100 rows x 4 columns]


In [19]:
drivers = []


for i in range(17, 47):
    drivers.append({
        "cancellation_odds": 0,
        "rating": 0,
        "user_id": i
    })

df = pd.DataFrame(drivers)

supabase.table("drivers").insert(drivers).execute()

print(df)

    cancellation_odds  rating  user_id
0                   0       0       17
1                   0       0       18
2                   0       0       19
3                   0       0       20
4                   0       0       21
5                   0       0       22
6                   0       0       23
7                   0       0       24
8                   0       0       25
9                   0       0       26
10                  0       0       27
11                  0       0       28
12                  0       0       29
13                  0       0       30
14                  0       0       31
15                  0       0       32
16                  0       0       33
17                  0       0       34
18                  0       0       35
19                  0       0       36
20                  0       0       37
21                  0       0       38
22                  0       0       39
23                  0       0       40
24                  0    

In [ ]:
riders = []


for i in range(47, 117):
    riders.append({
        "cancellation_odds": 0,
        "rating": 0,
        "user_id": i
    })

df = pd.DataFrame(riders)

supabase.table("riders").insert(riders).execute()

print(df)

   cancellation_odds  rating  user_id
0                  0       0      108
1                  0       0      109
2                  0       0      110
3                  0       0      111
4                  0       0      112
5                  0       0      113
6                  0       0      114
7                  0       0      115
8                  0       0      116


In [24]:
random.seed(42)

drivers = list(range(6, 36))

brands_models = {
    "Toyota": ["Corolla", "Yaris", "Hilux"],
    "Chevrolet": ["Spark", "Onix", "Tracker"],
    "Renault": ["Logan", "Sandero", "Duster"],
    "Kia": ["Picanto", "Rio", "Sportage"],
    "Mazda": ["Mazda 2", "Mazda 3", "CX-5"],
    "Hyundai": ["i10", "Accent", "Tucson"]
}

colors = [
    "White",
    "Black",
    "Gray",
    "Silver",
    "Blue",
    "Red"
]

used_plates = set()

def generate_plate():
    while True:
        letters = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=3))
        numbers = ''.join(random.choices('0123456789', k=3))
        plate = f"{letters}{numbers}"

        if plate not in used_plates:
            used_plates.add(plate)
            return plate

vehicles = []

for driver_id in drivers:

    brand = random.choice(list(brands_models.keys()))
    model = random.choice(brands_models[brand])

    vehicles.append({
        "brand": brand,
        "model": model,
        "color": random.choice(colors),
        "license_plate": generate_plate(),
        "number_slots": random.randint(3, 6),
        "driver_id": driver_id
    })
    
supabase.table("vehicles").insert(vehicles).execute()

vehicles_df = pd.DataFrame(vehicles)

print(vehicles_df.head())

     brand     model   color license_plate  number_slots  driver_id
0  Hyundai       i10   White        TGD175             6          6
1   Toyota   Corolla   White        FNA165             4          7
2      Kia  Sportage    Gray        VAU631             5          8
3   Toyota   Corolla  Silver        CWP875             6          9
4   Toyota     Hilux    Gray        VQW570             4         10


In [ ]:


random.seed(42)

driver_vehicle_map = {
    row["driver_id"]: row["driver_id"]
    for _, row in vehicles_df.iterrows()
}

# -----------------------------------
# Generar rides reutilizables
# Un ride puede transportar varios riders
# -----------------------------------

rides = []
ride_assignments = []

ride_id_counter = 1

# agrupamos por driver
for driver_id, group in df.groupby("driver_id"):

    riders_for_driver = group["rider_id"].tolist()

    # cantidad razonable de rides por driver
    n_rides = random.randint(8, 20)

    driver_rides = []

    for _ in range(n_rides):

        ride_type = random.choice([
            "TO_UNIVERSITY",
            "FROM_UNIVERSITY"
        ])

        zone_id = random.randint(1, 3)

        if ride_type == "TO_UNIVERSITY":
            source = f"Zona {zone_id}"
            destination = "Universidad de Los Andes"
        else:
            source = "Universidad de Los Andes"
            destination = f"Zona {zone_id}"

        ride_date = (
            datetime(2025, 1, 1) +
            timedelta(days=random.randint(0, 120))
        ).strftime("%Y-%m-%d")

        departure_time = (
            datetime(2025, 1, 1, 5, 0, 0) +
            timedelta(minutes=random.randint(0, 900))
        ).strftime("%H:%M:%S")

        ride = {
            "driver_id": driver_id,
            "vehicle_id": driver_vehicle_map[driver_id],
            "zone_id": zone_id,
            "type": ride_type,
            "source": source,
            "destination": destination,
            "price": random.randint(4000, 6000),
            "date": ride_date,
            "departure_time": departure_time,
            "state": random.choices(
                ["FINALIZADO", "OFERTADO"],
                weights=[0.8, 0.2]
            )[0]
        }

        rides.append(ride)
        driver_rides.append(ride_id_counter)

        ride_id_counter += 1

    # -----------------------------------
    # Asignar cada combinación rider-driver
    # a uno de los rides del driver
    # -----------------------------------

    for rider_id in riders_for_driver:

        assigned_ride = random.choice(driver_rides)

        ride_assignments.append({
            "rider_id": rider_id,
            "driver_id": driver_id,
            "ride_id": assigned_ride
        })

# -----------------------------------
# DataFrames finales
# -----------------------------------

rides_df = pd.DataFrame(rides)

ride_assignments_df = pd.DataFrame(ride_assignments)

print(rides_df.head())
print()
print(ride_assignments_df.head())

   driver_id  vehicle_id  zone_id             type                    source  \
0          6           6        1    TO_UNIVERSITY                    Zona 1   
1          6           6        3    TO_UNIVERSITY                    Zona 3   
2          6           6        1    TO_UNIVERSITY                    Zona 1   
3          6           6        3    TO_UNIVERSITY                    Zona 3   
4          6           6        1  FROM_UNIVERSITY  Universidad de Los Andes   

                destination  price        date departure_time       state  
0  Universidad de Los Andes   4501  2025-04-05       09:41:00  FINALIZADO  
1  Universidad de Los Andes   4178  2025-04-05       14:18:00  FINALIZADO  
2  Universidad de Los Andes   4476  2025-01-12       08:43:00  FINALIZADO  
3  Universidad de Los Andes   5330  2025-01-26       17:13:00  FINALIZADO  
4                    Zona 1   4569  2025-02-27       15:03:00    OFERTADO  

   rider_id  driver_id  ride_id
0        50          6       1

In [29]:
supabase.table("rides").insert(rides).execute()

APIResponse(data=[{'id': 116, 'driver_id': 6, 'vehicle_id': 6, 'zone_id': 1, 'source': 'Zona 1', 'destination': 'Universidad de Los Andes', 'date': '2025-04-05', 'departure_time': '09:41:00', 'state': 'FINALIZADO', 'type': 'TO_UNIVERSITY', 'price': 4501}, {'id': 117, 'driver_id': 6, 'vehicle_id': 6, 'zone_id': 3, 'source': 'Zona 3', 'destination': 'Universidad de Los Andes', 'date': '2025-04-05', 'departure_time': '14:18:00', 'state': 'FINALIZADO', 'type': 'TO_UNIVERSITY', 'price': 4178}, {'id': 118, 'driver_id': 6, 'vehicle_id': 6, 'zone_id': 1, 'source': 'Zona 1', 'destination': 'Universidad de Los Andes', 'date': '2025-01-12', 'departure_time': '08:43:00', 'state': 'FINALIZADO', 'type': 'TO_UNIVERSITY', 'price': 4476}, {'id': 119, 'driver_id': 6, 'vehicle_id': 6, 'zone_id': 3, 'source': 'Zona 3', 'destination': 'Universidad de Los Andes', 'date': '2025-01-26', 'departure_time': '17:13:00', 'state': 'FINALIZADO', 'type': 'TO_UNIVERSITY', 'price': 5330}, {'id': 120, 'driver_id': 6, 'v

In [40]:
random.seed(42)
np.random.seed(42)

# -----------------------------------
# ride_assignments_df:
# columnas:
# rider_id
# driver_id
# ride_id
# -----------------------------------

# -----------------------------------
# Factores latentes
# -----------------------------------

drivers = ride_assignments_df["driver_id"].unique()
riders = ride_assignments_df["rider_id"].unique()

driver_quality = {
    d: np.random.normal(0, 1)
    for d in drivers
}

rider_preference = {
    r: np.random.normal(0, 1)
    for r in riders
}

# -----------------------------------
# Función de ratings realistas
# -----------------------------------

def generate_scores(rider_id, driver_id):

    latent_score = (
        rider_preference[rider_id] * 0.4 +
        driver_quality[driver_id] * 0.6
    )

    base = 3 + latent_score

    punctuality = np.clip(
        round(base + np.random.normal(0, 0.5)),
        1,
        5
    )

    behavior = np.clip(
        round(base + np.random.normal(0, 0.4)),
        1,
        5
    )

    communication = np.clip(
        round(base + np.random.normal(0, 0.6)),
        1,
        5
    )

    security = np.clip(
        round(base + np.random.normal(0, 0.3)),
        1,
        5
    )

    return (
        int(punctuality),
        int(behavior),
        int(communication),
        int(security)
    )

# -----------------------------------
# Crear combinaciones finales
# -----------------------------------

ratings_rows = []

for _, row in ride_assignments_df.iterrows():

    rider_id = row["rider_id"]
    driver_id = row["driver_id"]
    ride_id = row["ride_id"]

    punctuality, behavior, communication, security = generate_scores(
        rider_id,
        driver_id
    )

    ratings_rows.append({
        "rider_id": int(rider_id),
        "driver_id": int(driver_id),
        "ride_id": int(ride_id)+115,
        "punctuality": int(punctuality),
        "behavior": int(behavior),
        "communication": int(communication),
        "security": int(security)
    })

ratings_df = pd.DataFrame(ratings_rows)

print(ratings_df.head())

   rider_id  driver_id  ride_id  punctuality  behavior  communication  \
0        50          6      131            2         3              3   
1        63          6      128            4         4              5   
2        11          6      130            3         3              2   
3        73          6      120            3         4              3   
4        69          6      124            4         3              4   

   security  
0         3  
1         4  
2         3  
3         3  
4         4  


In [41]:
supabase.table("rates_driver").insert(ratings_rows).execute()

APIResponse(data=[{'id': 2817, 'rider_id': 50, 'driver_id': 6, 'punctuality': 2, 'behavior': 3, 'communication': 3, 'security': 3, 'ride_id': 131}, {'id': 2818, 'rider_id': 63, 'driver_id': 6, 'punctuality': 4, 'behavior': 4, 'communication': 5, 'security': 4, 'ride_id': 128}, {'id': 2819, 'rider_id': 11, 'driver_id': 6, 'punctuality': 3, 'behavior': 3, 'communication': 2, 'security': 3, 'ride_id': 130}, {'id': 2820, 'rider_id': 73, 'driver_id': 6, 'punctuality': 3, 'behavior': 4, 'communication': 3, 'security': 3, 'ride_id': 120}, {'id': 2821, 'rider_id': 69, 'driver_id': 6, 'punctuality': 4, 'behavior': 3, 'communication': 4, 'security': 4, 'ride_id': 124}, {'id': 2822, 'rider_id': 35, 'driver_id': 6, 'punctuality': 3, 'behavior': 2, 'communication': 4, 'security': 2, 'ride_id': 120}, {'id': 2823, 'rider_id': 77, 'driver_id': 6, 'punctuality': 4, 'behavior': 4, 'communication': 3, 'security': 3, 'ride_id': 123}, {'id': 2824, 'rider_id': 62, 'driver_id': 6, 'punctuality': 3, 'behavior

In [44]:
today = datetime.today()

i=116
for ride in rides:

    # Si está OFERTADO -> fecha futura
    if ride["state"] == "OFERTADO":

        future_date = today + timedelta(
            days=random.randint(1, 30)
        )

        ride["date"] = future_date.strftime("%Y-%m-%d")

    # Si está FINALIZADO -> fecha pasada
    else:

        past_date = today - timedelta(
            days=random.randint(1, 120)
        )

        ride["date"] = past_date.strftime("%Y-%m-%d")
    ride["id"] = i
    i += 1

# reconstruir dataframe
rides_df = pd.DataFrame(rides)

print(rides_df.head())

   driver_id  vehicle_id  zone_id             type                    source  \
0          6           6        1    TO_UNIVERSITY                    Zona 1   
1          6           6        3    TO_UNIVERSITY                    Zona 3   
2          6           6        1    TO_UNIVERSITY                    Zona 1   
3          6           6        3    TO_UNIVERSITY                    Zona 3   
4          6           6        1  FROM_UNIVERSITY  Universidad de Los Andes   

                destination  price        date departure_time       state   id  
0  Universidad de Los Andes   4501  2026-05-03       09:41:00  FINALIZADO  116  
1  Universidad de Los Andes   4178  2026-05-04       14:18:00  FINALIZADO  117  
2  Universidad de Los Andes   4476  2026-02-25       08:43:00  FINALIZADO  118  
3  Universidad de Los Andes   5330  2026-03-10       17:13:00  FINALIZADO  119  
4                    Zona 1   4569  2026-05-28       15:03:00    OFERTADO  120  


In [45]:
for _, row in rides_df.iterrows():

    supabase.table("rides").update({
        "date": row["date"]
    }).eq("id", int(row["id"])).execute()

print("Rides actualizados")

Rides actualizados
